# TP5 : Analyse de ventes e-commerce avec Spark SQL
**Écosystème Big Data 2** — Nettoyage, typage, jointures, agrégations, fonctions de fenêtre et optimisation Catalyst

## Partie 0 – Génération des données

In [1]:
import os, csv, json, random
from datetime import date, timedelta

random.seed(42)
os.makedirs("data_tp2", exist_ok=True)

catalogue = {
    "Informatique": ["Clavier", "Souris", "Ecran", "Casque", "Webcam"],
    "Maison":       ["Lampe", "Chaise", "Bureau", "Etagere"],
    "Sport":        ["Ballon", "Raquette", "Tapis de yoga", "Haltères"],
}
produits_l, pid = [], 1
for cat, noms in catalogue.items():
    for n in noms:
        produits_l.append([pid, n, cat, random.choice([49, 79, 99, 149, 249, 399, 599])])
        pid += 1
produits_l[3][3] = "N/A"
with open("data_tp2/produits.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_produit", "nom", "categorie", "prix"])
    w.writerows(produits_l)

villes = ["Casablanca", "Rabat", "Fès", "Marrakech", "Tanger", "Agadir"]
prenoms = ["Sara", "Youssef", "Imane", "Omar", "Salma",
           "Hamza", "Khadija", "Anas", "Meryem", "Ayoub"]
familles = ["Alaoui", "Benani", "Cherkaoui", "Darif", "El Idrissi", "Fassi", "Tazi"]
with open("data_tp2/clients.json", "w", encoding="utf-8") as f:
    for i in range(1, 61):
        ville = random.choice(villes)
        if i % 7 == 0:
            ville = " " + ville.upper() + " "
        c = {"id_client": i,
             "nom": random.choice(prenoms) + " " + random.choice(familles),
             "ville": ville,
             "date_inscription": (date(2024, 1, 1)
                                  + timedelta(days=random.randint(0, 364))).isoformat()}
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

statuts = ["livree", "livree", "livree", "annulee", "en_cours", "Livree", "LIVREE "]
cmd = []
for i in range(1, 3001):
    d = date(2025, 1, 1) + timedelta(days=random.randint(0, 364))
    cmd.append([i, random.randint(1, 54), random.randint(1, 13),
                random.randint(1, 5), d.isoformat(), random.choice(statuts)])
for k in range(0, 3000, 150):  cmd[k][3] = "abc"
for k in range(5, 3000, 200):  cmd[k][3] = -2
for k in range(10, 3000, 300): cmd[k][4] = ""
for k in range(20, 3000, 500): cmd[k][1] = ""
cmd += random.sample(cmd, 40)
with open("data_tp2/commandes.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_commande", "id_client", "id_produit",
                "quantite", "date_commande", "statut"])
    w.writerows(cmd)
print(sorted(os.listdir("data_tp2")))

['clients.json', 'commandes.csv', 'produits.csv']


## Partie 1 – Chargement avec schéma explicite

### Question 1.1 – SparkSession

In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("TP2 - Ventes")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/09 09:47:22 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/09 09:47:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/09 09:47:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.3.0'

### Question 1.2 – Chargement (tout en `StringType`)

In [3]:
produits_raw = (spark.read.option("header", True)
    .schema("id_produit STRING, nom STRING, categorie STRING, prix STRING")
    .csv("data_tp2/produits.csv"))
commandes_raw = (spark.read.option("header", True)
    .schema("id_commande STRING, id_client STRING, id_produit STRING, quantite STRING, date_commande STRING, statut STRING")
    .csv("data_tp2/commandes.csv"))
clients_raw = spark.read.json("data_tp2/clients.json")

for nom, df in [("produits_raw", produits_raw), ("clients_raw", clients_raw), ("commandes_raw", commandes_raw)]:
    print("=====", nom, "=====")
    df.printSchema()
    df.show(5, truncate=False)
    print("Nombre de lignes :", df.count(), "\n")

===== produits_raw =====
root
 |-- id_produit: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix: string (nullable = true)



+----------+-------+------------+----+
|id_produit|nom    |categorie   |prix|
+----------+-------+------------+----+
|1         |Clavier|Informatique|399 |
|2         |Souris |Informatique|49  |
|3         |Ecran  |Informatique|49  |
|4         |Casque |Informatique|N/A |
|5         |Webcam |Informatique|99  |
+----------+-------+------------+----+
only showing top 5 rows


Nombre de lignes : 13 

===== clients_raw =====
root
 |-- date_inscription: string (nullable = true)
 |-- id_client: long (nullable = true)
 |-- nom: string (nullable = true)
 |-- ville: string (nullable = true)



+----------------+---------+---------------+----------+
|date_inscription|id_client|nom            |ville     |
+----------------+---------+---------------+----------+
|2024-01-17      |1        |Ayoub Darif    |Casablanca|
|2024-04-29      |2        |Youssef Benani |Casablanca|
|2024-10-14      |3        |Ayoub Alaoui   |Tanger    |
|2024-04-22      |4        |Meryem Darif   |Rabat     |
|2024-01-04      |5        |Ayoub Cherkaoui|Marrakech |
+----------------+---------+---------------+----------+
only showing top 5 rows


Nombre de lignes : 60 

===== commandes_raw =====
root
 |-- id_commande: string (nullable = true)
 |-- id_client: string (nullable = true)
 |-- id_produit: string (nullable = true)
 |-- quantite: string (nullable = true)
 |-- date_commande: string (nullable = true)
 |-- statut: string (nullable = true)



+-----------+---------+----------+--------+-------------+-------+
|id_commande|id_client|id_produit|quantite|date_commande|statut |
+-----------+---------+----------+--------+-------------+-------+
|1          |51       |8         |abc     |2025-05-05   |livree |
|2          |7        |11        |4       |2025-02-18   |livree |
|3          |27       |8         |1       |2025-08-05   |Livree |
|4          |42       |2         |1       |2025-12-01   |annulee|
|5          |52       |2         |2       |2025-06-23   |livree |
+-----------+---------+----------+--------+-------------+-------+
only showing top 5 rows


Nombre de lignes : 3040 



## Partie 2 – Audit de la qualité des données

### Question 2.1 – Valeurs manquantes (null ou chaîne vide)

In [4]:
commandes_raw.select([
    F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1)).alias(c)
    for c in commandes_raw.columns
]).show()

+-----------+---------+----------+--------+-------------+------+
|id_commande|id_client|id_produit|quantite|date_commande|statut|
+-----------+---------+----------+--------+-------------+------+
|          0|        7|         0|       0|           10|     0|
+-----------+---------+----------+--------+-------------+------+



### Question 2.2 – Doublons et statuts

In [5]:
print("Doublons exacts :", commandes_raw.count() - commandes_raw.dropDuplicates().count())
commandes_raw.groupBy("statut").count().orderBy("statut").show()

Doublons exacts : 40


+--------+-----+
|  statut|count|
+--------+-----+
| LIVREE |  426|
|  Livree|  465|
| annulee|  476|
|en_cours|  413|
|  livree| 1260|
+--------+-----+



**Remarque :** on observe 5 valeurs distinctes de `statut` alors qu'il n'y a que 3 statuts réels (livrée, annulée, en cours). `livree`, `Livree` et `LIVREE ` désignent la même valeur : problèmes de casse et d'espaces. Sans normalisation, le CA serait sous-estimé (seul `livree` serait compté).

### Question 2.3 – Quantités non numériques ou négatives

In [6]:
q = F.col("quantite").try_cast("int")
invalides = commandes_raw.filter(q.isNull() | (q <= 0))
print("Lignes concernées :", invalides.count())
invalides.show(5)

Lignes concernées : 35


+-----------+---------+----------+--------+-------------+-------+
|id_commande|id_client|id_produit|quantite|date_commande| statut|
+-----------+---------+----------+--------+-------------+-------+
|          1|       51|         8|     abc|   2025-05-05| livree|
|          6|       35|         8|      -2|   2025-04-08|annulee|
|        151|       25|         8|     abc|   2025-09-25| livree|
|        206|       44|         5|      -2|   2025-11-20| Livree|
|        301|       43|         5|     abc|   2025-09-27| livree|
+-----------+---------+----------+--------+-------------+-------+
only showing top 5 rows


### Question 2.4 – Problème de la colonne `prix`

In [7]:
produits_raw.filter(F.col("prix").try_cast("int").isNull()).show()

+----------+------+------------+----+
|id_produit|   nom|   categorie|prix|
+----------+------+------------+----+
|         4|Casque|Informatique| N/A|
+----------+------+------------+----+



**Réponse :** la colonne `prix` contient la valeur non numérique `N/A` pour le produit 4 (**Casque**, catégorie Informatique). Elle empêche tout calcul de CA pour ce produit.

## Partie 3 – Nettoyage et typage

### Question 3.1 – DataFrame `commandes` (R1 à R4)

In [8]:
n_avant = commandes_raw.count()

typees = (commandes_raw
    .dropDuplicates()                                                        # R1
    .select(F.col("id_commande").try_cast("int").alias("id_commande"),       # R2
            F.col("id_client").try_cast("int").alias("id_client"),
            F.col("id_produit").try_cast("int").alias("id_produit"),
            F.col("quantite").try_cast("int").alias("quantite"),
            F.col("date_commande").try_cast("date").alias("date_commande"),
            F.lower(F.trim(F.col("statut"))).alias("statut")))               # R3

commandes = typees.filter(                                                   # R4 (après typage)
    F.col("quantite").isNotNull() & (F.col("quantite") > 0) &
    F.col("date_commande").isNotNull() & F.col("id_client").isNotNull())

print("Avant :", n_avant, "| Après :", commandes.count())
commandes.printSchema()
commandes.groupBy("statut").count().show()

Avant : 3040 | Après : 2949
root
 |-- id_commande: integer (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- id_produit: integer (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- date_commande: date (nullable = true)
 |-- statut: string (nullable = true)



+--------+-----+
|  statut|count|
+--------+-----+
|en_cours|  397|
| annulee|  463|
|  livree| 2089|
+--------+-----+



### Question 3.2 – DataFrame `produits` (R2 et R5)

In [9]:
w = Window.partitionBy("categorie")
produits = (produits_raw
    .select(F.col("id_produit").try_cast("int").alias("id_produit"), "nom", "categorie",
            F.col("prix").try_cast("double").alias("prix"))
    .withColumn("prix", F.round(F.coalesce(F.col("prix"), F.avg("prix").over(w)), 2)))
produits.orderBy("id_produit").show()

+----------+-------------+------------+-----+
|id_produit|          nom|   categorie| prix|
+----------+-------------+------------+-----+
|         1|      Clavier|Informatique|399.0|
|         2|       Souris|Informatique| 49.0|
|         3|        Ecran|Informatique| 49.0|
|         4|       Casque|Informatique|149.0|
|         5|       Webcam|Informatique| 99.0|
|         6|        Lampe|      Maison| 79.0|
|         7|       Chaise|      Maison| 79.0|
|         8|       Bureau|      Maison| 79.0|
|         9|      Etagere|      Maison|399.0|
|        10|       Ballon|       Sport| 49.0|
|        11|     Raquette|       Sport|399.0|
|        12|Tapis de yoga|       Sport|399.0|
|        13|     Haltères|       Sport|249.0|
+----------+-------------+------------+-----+



### Question 3.3 – DataFrame `clients` (R2 et R3)

In [10]:
clients = clients_raw.select(
    F.col("id_client").try_cast("int").alias("id_client"),
    "nom",
    F.initcap(F.trim(F.col("ville"))).alias("ville"),
    F.col("date_inscription").try_cast("date").alias("date_inscription"))

clients.select("ville").distinct().show()
print("Villes distinctes :", clients.select("ville").distinct().count())

+----------+
|     ville|
+----------+
|     Rabat|
| Marrakech|
|Casablanca|
|       Fès|
|    Tanger|
|    Agadir|
+----------+



Villes distinctes : 6


## Partie 4 – Requêtes SQL sur vues temporaires

### Question 4.1 – Vues et CA total

In [11]:
produits.createOrReplaceTempView("produits")
clients.createOrReplaceTempView("clients")
commandes.createOrReplaceTempView("commandes")

spark.sql("""
SELECT SUM(c.quantite * p.prix) AS ca_total, COUNT(*) AS nb_commandes
FROM commandes c JOIN produits p ON c.id_produit = p.id_produit
WHERE c.statut = 'livree'
""").show()

+---------+------------+
| ca_total|nb_commandes|
+---------+------------+
|1215123.0|        2089|
+---------+------------+



### Question 4.2 – CA par catégorie

In [12]:
spark.sql("""
SELECT p.categorie,
       ROUND(SUM(c.quantite * p.prix), 2) AS ca,
       COUNT(*) AS nb_commandes,
       ROUND(SUM(c.quantite * p.prix) / COUNT(*), 2) AS panier_moyen
FROM commandes c JOIN produits p ON c.id_produit = p.id_produit
WHERE c.statut = 'livree'
GROUP BY p.categorie
ORDER BY ca DESC
""").show()

+------------+--------+------------+------------+
|   categorie|      ca|nb_commandes|panier_moyen|
+------------+--------+------------+------------+
|       Sport|574687.0|         685|      838.96|
|Informatique|354311.0|         800|      442.89|
|      Maison|286125.0|         604|      473.72|
+------------+--------+------------+------------+



### Question 4.3 – CA mensuel

In [13]:
ca_mensuel = spark.sql("""
SELECT date_format(c.date_commande, 'yyyy-MM') AS mois,
       ROUND(SUM(c.quantite * p.prix), 2) AS ca
FROM commandes c JOIN produits p ON c.id_produit = p.id_produit
WHERE c.statut = 'livree'
GROUP BY 1 ORDER BY 1
""")
ca_mensuel.show(12)
print("Mois au CA le plus élevé :")
ca_mensuel.orderBy(F.desc("ca")).show(1)

+-------+--------+
|   mois|      ca|
+-------+--------+
|2025-01|118017.0|
|2025-02| 98790.0|
|2025-03|109451.0|
|2025-04| 81453.0|
|2025-05| 86787.0|
|2025-06|114252.0|
|2025-07|106081.0|
|2025-08|114002.0|
|2025-09| 92077.0|
|2025-10| 90417.0|
|2025-11| 99390.0|
|2025-12|104406.0|
+-------+--------+

Mois au CA le plus élevé :


+-------+--------+
|   mois|      ca|
+-------+--------+
|2025-01|118017.0|
+-------+--------+
only showing top 1 row


### Question 4.4 – Top 5 produits

In [14]:
spark.sql("""
SELECT p.nom, p.categorie, SUM(c.quantite) AS qte_totale,
       ROUND(SUM(c.quantite * p.prix), 2) AS ca
FROM commandes c JOIN produits p ON c.id_produit = p.id_produit
WHERE c.statut = 'livree'
GROUP BY p.nom, p.categorie
ORDER BY ca DESC LIMIT 5
""").show()

+-------------+------------+----------+--------+
|          nom|   categorie|qte_totale|      ca|
+-------------+------------+----------+--------+
|     Raquette|       Sport|       552|220248.0|
|Tapis de yoga|       Sport|       517|206283.0|
|      Clavier|Informatique|       491|195909.0|
|      Etagere|      Maison|       451|179949.0|
|     Haltères|       Sport|       485|120765.0|
+-------------+------------+----------+--------+



## Partie 5 – Jointures en API DataFrame

### Question 5.1 – DataFrame `ventes`

In [15]:
ventes = (commandes.filter(F.col("statut") == "livree")
    .join(produits.withColumnRenamed("nom", "produit"), "id_produit")
    .join(clients.withColumnRenamed("nom", "nom_client"), "id_client")
    .select("id_commande", "date_commande", "nom_client", "ville",
            F.col("produit").alias("nom"), "categorie", "quantite", "prix",
            (F.col("quantite") * F.col("prix")).alias("ca")))
ventes.show(5)
print("Lignes :", ventes.count())

+-----------+-------------+------------------+----------+-------------+------------+--------+-----+------+
|id_commande|date_commande|        nom_client|     ville|          nom|   categorie|quantite| prix|    ca|
+-----------+-------------+------------------+----------+-------------+------------+--------+-----+------+
|          7|   2025-04-04|       Salma Fassi|     Rabat|       Bureau|      Maison|       2| 79.0| 158.0|
|         14|   2025-09-07|   Anas El Idrissi|Casablanca|       Casque|Informatique|       3|149.0| 447.0|
|         15|   2025-01-30|       Imane Darif| Marrakech|Tapis de yoga|       Sport|       5|399.0|1995.0|
|         24|   2025-12-10|        Hamza Tazi|     Rabat|       Webcam|Informatique|       4| 99.0| 396.0|
|         30|   2025-12-01|Khadija El Idrissi| Marrakech|      Clavier|Informatique|       5|399.0|1995.0|
+-----------+-------------+------------------+----------+-------------+------------+--------+-----+------+
only showing top 5 rows


Lignes : 2089


### Question 5.2 – CA par ville et vérification

In [16]:
ca_ville = (ventes.groupBy("ville").agg(F.round(F.sum("ca"), 2).alias("ca"))
            .orderBy(F.desc("ca")))
ca_ville.show()
total_villes = ventes.agg(F.sum("ca")).first()[0]
print("Somme des CA par ville :", total_villes)

+----------+--------+
|     ville|      ca|
+----------+--------+
|     Rabat|253369.0|
|    Tanger|220226.0|
|       Fès|209395.0|
| Marrakech|208387.0|
|    Agadir|170305.0|
|Casablanca|153441.0|
+----------+--------+



Somme des CA par ville : 1215123.0


### Question 5.3 – Clients sans commande (`left_anti`)

In [17]:
sans_commande = clients.join(commandes, "id_client", "left_anti")
print("Clients sans commande :", sans_commande.count())
sans_commande.orderBy("id_client").show()

Clients sans commande : 6


+---------+------------------+----------+----------------+
|id_client|               nom|     ville|date_inscription|
+---------+------------------+----------+----------------+
|       55|   Omar El Idrissi|    Tanger|      2024-04-22|
|       56|     Youssef Fassi|Casablanca|      2024-11-19|
|       57|       Omar Alaoui|Casablanca|      2024-01-17|
|       58|Youssef El Idrissi|       Fès|      2024-05-01|
|       59|       Anas Benani|       Fès|      2024-10-03|
|       60|  Ayoub El Idrissi|     Rabat|      2024-08-30|
+---------+------------------+----------+----------------+



### Question 5.4 – Types de jointures

**Réponse :**
- **inner** : ne garde que les lignes ayant une correspondance des deux côtés.
- **left** : garde toutes les lignes de la table de gauche ; les colonnes de droite valent `null` en l'absence de correspondance.
- **left_semi** : garde les lignes de gauche qui ont une correspondance à droite, sans ajouter de colonnes de droite (équivalent d'un `EXISTS`).
- **left_anti** : garde les lignes de gauche qui n'ont **aucune** correspondance à droite (équivalent d'un `NOT EXISTS`).

En 5.1, un `left join` donnerait un résultat différent de l'`inner join` s'il existait des commandes dont le `id_produit` ou le `id_client` n'a pas de correspondance dans `produits` ou `clients` : elles seraient conservées avec des `null` (et un CA `null`) au lieu d'être éliminées. Ici, la vérification ci-dessous donne 0 cas, donc les deux jointures sont équivalentes.

In [18]:
print("Commandes sans produit :", commandes.join(produits, "id_produit", "left_anti").count())
print("Commandes sans client   :", commandes.join(clients, "id_client", "left_anti").count())

Commandes sans produit : 0


Commandes sans client   : 0


## Partie 6 – Fonctions de fenêtre

### Question 6.1 – Top 2 produits par catégorie (`dense_rank`)

In [19]:
ca_produit = ventes.groupBy("categorie", "nom").agg(F.round(F.sum("ca"), 2).alias("ca"))
w_rang = Window.partitionBy("categorie").orderBy(F.desc("ca"))
(ca_produit.withColumn("rang", F.dense_rank().over(w_rang))
           .filter(F.col("rang") <= 2)
           .orderBy("categorie", "rang").show())

+------------+-------------+--------+----+
|   categorie|          nom|      ca|rang|
+------------+-------------+--------+----+
|Informatique|      Clavier|195909.0|   1|
|Informatique|       Casque| 60941.0|   2|
|      Maison|      Etagere|179949.0|   1|
|      Maison|       Bureau| 39184.0|   2|
|       Sport|     Raquette|220248.0|   1|
|       Sport|Tapis de yoga|206283.0|   2|
+------------+-------------+--------+----+



### Question 6.2 – CA cumulé et variation mensuelle (`lag`)

In [20]:
ca_m = (ventes.withColumn("mois", F.date_format("date_commande", "yyyy-MM"))
              .groupBy("mois").agg(F.sum("ca").alias("ca")))
w_m = Window.orderBy("mois")
(ca_m.select("mois",
             F.round("ca", 2).alias("ca"),
             F.round(F.sum("ca").over(w_m.rowsBetween(Window.unboundedPreceding, Window.currentRow)), 2).alias("ca_cumule"),
             F.round((F.col("ca") - F.lag("ca", 1).over(w_m)) / F.lag("ca", 1).over(w_m) * 100, 2).alias("variation_pct"))
     .orderBy("mois").show(12))

+-------+--------+---------+-------------+
|   mois|      ca|ca_cumule|variation_pct|
+-------+--------+---------+-------------+
|2025-01|118017.0| 118017.0|         NULL|
|2025-02| 98790.0| 216807.0|       -16.29|
|2025-03|109451.0| 326258.0|        10.79|
|2025-04| 81453.0| 407711.0|       -25.58|
|2025-05| 86787.0| 494498.0|         6.55|
|2025-06|114252.0| 608750.0|        31.65|
|2025-07|106081.0| 714831.0|        -7.15|
|2025-08|114002.0| 828833.0|         7.47|
|2025-09| 92077.0| 920910.0|       -19.23|
|2025-10| 90417.0|1011327.0|         -1.8|
|2025-11| 99390.0|1110717.0|         9.92|
|2025-12|104406.0|1215123.0|         5.05|
+-------+--------+---------+-------------+



## Partie 7 – Segmentation des clients

### Question 7.1 – API DataFrame

In [21]:
# Agrégation par id_client (le nom n'est pas unique : il existe des homonymes)
ca_client = (commandes.filter("statut = 'livree'").join(produits, "id_produit")
             .groupBy("id_client").agg(F.sum(F.col("quantite") * F.col("prix")).alias("ca")))

segments = ca_client.withColumn("segment",
    F.when(F.col("ca") >= 25000, "Gold")
     .when(F.col("ca") >= 20000, "Silver")
     .otherwise("Bronze"))
segments.groupBy("segment").count().show()

+-------+-----+
|segment|count|
+-------+-----+
|   Gold|   16|
| Bronze|   22|
| Silver|   16|
+-------+-----+



### Question 7.2 – SQL pur (`CASE WHEN`) et top 5 clients

In [22]:
spark.sql("""
WITH t AS (
  SELECT c.id_client,
         SUM(c.quantite * p.prix) AS ca,
         CASE WHEN SUM(c.quantite * p.prix) >= 25000 THEN 'Gold'
              WHEN SUM(c.quantite * p.prix) >= 20000 THEN 'Silver'
              ELSE 'Bronze' END AS segment
  FROM commandes c JOIN produits p ON c.id_produit = p.id_produit
  WHERE c.statut = 'livree'
  GROUP BY c.id_client)
SELECT cl.nom, cl.ville, ROUND(t.ca, 2) AS ca, t.segment
FROM t JOIN clients cl ON t.id_client = cl.id_client
ORDER BY t.ca DESC LIMIT 5
""").show()

+-----------------+------+-------+-------+
|              nom| ville|     ca|segment|
+-----------------+------+-------+-------+
|     Hamza Alaoui|   Fès|41736.0|   Gold|
|Meryem El Idrissi|   Fès|38512.0|   Gold|
|Khadija Cherkaoui| Rabat|35376.0|   Gold|
|    Meryem Alaoui|Tanger|32523.0|   Gold|
| Salma El Idrissi|Agadir|31485.0|   Gold|
+-----------------+------+-------+-------+



## Partie 8 – Optimisation et plans d'exécution (Catalyst)

### Question 8.1 – Prédicat pushdown

In [23]:
commandes.join(produits, "id_produit").filter(F.col("categorie") == "Sport").explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (17)
+- Project (16)
   +- BroadcastHashJoin Inner BuildRight (15)
      :- HashAggregate (5)
      :  +- Exchange (4)
      :     +- HashAggregate (3)
      :        +- Filter (2)
      :           +- Scan csv  (1)
      +- BroadcastExchange (14)
         +- Project (13)
            +- Filter (12)
               +- Window (11)
                  +- Sort (10)
                     +- Exchange (9)
                        +- Project (8)
                           +- Filter (7)
                              +- Scan csv  (6)


(1) Scan csv 
Output [6]: [id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9]
Batched: false
Location: InMemoryFileIndex [file:/home/claude/tp/data_tp2/commandes.csv]
PushedFilters: [IsNotNull(quantite), IsNotNull(date_commande), IsNotNull(id_client)]
ReadSchema: struct<id_commande:string,id_client:string,id_produit:string,quantite:string,date_commande:string,statut:string>

(2) Filter
Input [6]: [id_c

**Réponse :** bien que le filtre soit écrit *après* la jointure, le plan optimisé le place **avant** la jointure, au niveau du scan de `produits` (voir `PushedFilters: [EqualTo(categorie,Sport)]`). Cette optimisation de Catalyst s'appelle le **predicate pushdown** (poussée des prédicats) : on réduit le volume de données avant la jointure.

### Question 8.2 – Stratégies de jointure et broadcast

In [24]:
print(">>> Plan par défaut :")
commandes.join(produits, "id_produit").explain()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print(">>> Diffusion automatique désactivée :")
commandes.join(produits, "id_produit").explain()

print(">>> Broadcast forcé :")
commandes.join(F.broadcast(produits), "id_produit").explain()

spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

>>> Plan par défaut :
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id_produit#278, id_commande#276, id_client#277, quantite#279, date_commande#280, statut#281, nom#1, categorie#2, prix#323]
   +- BroadcastHashJoin [id_produit#278], [id_produit#321], Inner, BuildRight, false, false
      :- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
      :  +- Exchange hashpartitioning(id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9, 8), ENSURE_REQUIREMENTS, [plan_id=7250]
      :     +- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
      :        +- Filter (((((((isnotnull(quantite#7) AND isnotnull(date_commande#8)) AND isnotnull(id_client#5)) AND isnotnull(try_cast(quantite#7 as int))) AND (try_cast(quantite#7 as int) > 0)) AND isnotnull(try_cast(date_commande#8 as date))) AND isnotnull(try_cast(id_client#

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id_produit#278, id_commande#276, id_client#277, quantite#279, date_commande#280, statut#281, nom#1, categorie#2, prix#323]
   +- BroadcastHashJoin [id_produit#278], [id_produit#321], Inner, BuildRight, false, false
      :- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
      :  +- Exchange hashpartitioning(id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9, 8), ENSURE_REQUIREMENTS, [plan_id=7370]
      :     +- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
      :        +- Filter (((((((isnotnull(quantite#7) AND isnotnull(date_commande#8)) AND isnotnull(id_client#5)) AND isnotnull(try_cast(quantite#7 as int))) AND (try_cast(quantite#7 as int) > 0)) AND isnotnull(try_cast(date_commande#8 as date))) AND isnotnull(try_cast(id_client#5 as int))) AND isnotn

**Réponse :** sans diffusion, Spark choisit un **SortMergeJoin**, qui impose un shuffle (`Exchange`) et un tri des deux côtés. Avec `F.broadcast`, il utilise un **BroadcastHashJoin** : la petite table `produits` (13 lignes) est envoyée à tous les exécuteurs et la grande table `commandes` n'est pas redistribuée. Le broadcast est efficace ici parce que `produits` est minuscule : le coût du shuffle disparaît.

### Question 8.3 – Effet du cache

In [25]:
import time

def chronometre(n=3):
    durees = []
    for _ in range(n):
        t = time.time()
        ventes.groupBy("ville").agg(F.sum("ca")).collect()
        durees.append(round(time.time() - t, 3))
    return durees

print("Sans cache :", chronometre())
ventes.cache()
ventes.count()                       # première action : remplit le cache
print("Avec cache :", chronometre())
ventes.unpersist()

Sans cache : [0.664, 0.472, 0.47]


Avec cache : [0.704, 0.263, 0.248]


DataFrame[id_commande: int, date_commande: date, nom_client: string, ville: string, nom: string, categorie: string, quantite: int, prix: double, ca: double]

**Constat :** les temps diminuent avec le cache, car `ventes` n'est plus recalculée (lecture CSV, nettoyage, jointures) à chaque action. Le gain est modéré sur de petites données, mais le principe reste valable ; l'onglet *Storage* de l'interface Spark (http://localhost:4040) montre les données mises en cache.

### Question 8.4 – DataFrame vs SQL

In [26]:
print(">>> DataFrame")
commandes.join(produits, "id_produit").groupBy("categorie").agg(F.sum("quantite")).explain()
print(">>> SQL")
spark.sql("""SELECT p.categorie, SUM(c.quantite)
FROM commandes c JOIN produits p ON c.id_produit = p.id_produit
GROUP BY p.categorie""").explain()

>>> DataFrame
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[categorie#2], functions=[sum(quantite#279)])
   +- Exchange hashpartitioning(categorie#2, 8), ENSURE_REQUIREMENTS, [plan_id=9279]
      +- HashAggregate(keys=[categorie#2], functions=[partial_sum(quantite#279)])
         +- Project [quantite#279, categorie#2]
            +- BroadcastHashJoin [id_produit#278], [id_produit#321], Inner, BuildRight, false, false
               :- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
               :  +- Exchange hashpartitioning(id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9, 8), ENSURE_REQUIREMENTS, [plan_id=9271]
               :     +- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
               :        +- Filter (((((((isnotnull(quantite#7) AND isnotnull(date_commande#8)) AND isnot

**Réponse :** les deux plans physiques sont identiques. Les deux API produisent le même plan logique, que Catalyst optimise de la même façon. Le choix de l'API ne change donc pas la performance : il dépend de la lisibilité, de la composabilité (DataFrame) ou de la familiarité de l'équipe (SQL).

## Partie 9 – Persistance et partitionnement

### Question 9.1 – Parquet partitionné par mois

In [27]:
ventes_p = (ventes.withColumn("annee", F.year("date_commande"))
                  .withColumn("mois", F.month("date_commande")))
ventes_p.write.mode("overwrite").partitionBy("mois").parquet("out_tp2/ventes")

import os
for d in sorted(os.listdir("out_tp2/ventes")):
    print(d)

._SUCCESS.crc
_SUCCESS
mois=1
mois=10
mois=11
mois=12
mois=2
mois=3
mois=4
mois=5
mois=6
mois=7
mois=8
mois=9


### Question 9.2 – Partition pruning vs pushdown

In [28]:
relu = spark.read.parquet("out_tp2/ventes")
print(">>> Filtre sur mois (colonne partitionnée)")
relu.filter("mois = 3").explain()
print(">>> Filtre sur ville (colonne non partitionnée)")
relu.filter("ville = 'Rabat'").explain()

>>> Filtre sur mois (colonne partitionnée)
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [id_commande#2093,date_commande#2094,nom_client#2095,ville#2096,nom#2097,categorie#2098,quantite#2099,prix#2100,ca#2101,annee#2102,mois#2103] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/claude/tp/out_tp2/ventes], PartitionFilters: [isnotnull(mois#2103), (mois#2103 = 3)], PushedFilters: [], ReadSchema: struct<id_commande:int,date_commande:date,nom_client:string,ville:string,nom:string,categorie:str...


>>> Filtre sur ville (colonne non partitionnée)
== Physical Plan ==
*(1) Filter (isnotnull(ville#2096) AND (ville#2096 = Rabat))
+- *(1) ColumnarToRow
   +- FileScan parquet [id_commande#2093,date_commande#2094,nom_client#2095,ville#2096,nom#2097,categorie#2098,quantite#2099,prix#2100,ca#2101,annee#2102,mois#2103] Batched: true, DataFilters: [isnotnull(ville#2096), (ville#2096 = Rabat)], Format: Parquet, Location: InMemoryFileIndex(1 p

**Réponse :** pour `mois = 3`, le plan affiche `PartitionFilters: [mois = 3]` : Spark ne lit que le dossier `mois=3` (**partition pruning**) et ignore les 11 autres. Pour `ville`, il n'y a que `PushedFilters: [EqualTo(ville,Rabat)]` : tous les fichiers sont ouverts et le filtre est appliqué à l'intérieur de Parquet (via les statistiques des groupes de lignes), ce qui est moins efficace.

### Question 9.3 – Rapport final CSV unique

In [29]:
rapport = (ventes_p.groupBy("categorie", "mois")
           .agg(F.round(F.sum("ca"), 2).alias("ca"))
           .orderBy("categorie", "mois"))
rapport.coalesce(1).write.mode("overwrite").option("header", True).csv("out_tp2/rapport_ca")

import glob
fichier = glob.glob("out_tp2/rapport_ca/part-*.csv")[0]
print(open(fichier).read())

categorie,mois,ca
Informatique,1,35488.0
Informatique,2,26147.0
Informatique,3,29241.0
Informatique,4,27263.0
Informatique,5,30193.0
Informatique,6,24902.0
Informatique,7,31906.0
Informatique,8,42962.0
Informatique,9,25664.0
Informatique,10,30399.0
Informatique,11,25374.0
Informatique,12,24772.0
Maison,1,26724.0
Maison,2,24820.0
Maison,3,25377.0
Maison,4,17389.0
Maison,5,21075.0
Maison,6,26957.0
Maison,7,27648.0
Maison,8,18989.0
Maison,9,20767.0
Maison,10,20265.0
Maison,11,25353.0
Maison,12,30761.0
Sport,1,55805.0
Sport,2,47823.0
Sport,3,54833.0
Sport,4,36801.0
Sport,5,35519.0
Sport,6,62393.0
Sport,7,46527.0
Sport,8,52051.0
Sport,9,45646.0
Sport,10,39753.0
Sport,11,48663.0
Sport,12,48873.0



### Question 9.4 – `coalesce(1)`

**Réponse :** `coalesce(1)` regroupe toutes les données dans **une seule partition**, donc une seule tâche écrit le fichier. Sur de gros volumes, cela supprime le parallélisme (goulot d'étranglement), peut saturer la mémoire d'un seul exécuteur (risque d'`OutOfMemory`) et ralentit fortement l'écriture. À réserver aux petits résultats comme ce rapport agrégé.

In [30]:
spark.stop()